# From MicroHH output to a SPECTRA plume column

SPECTRA embeds large-eddy-simulation (LES) plumes that come as
`PlumeColumn` files: one LES case, one tracer and one source as a time
series of vertical columns [kg m-2] on the native model grid. This
tutorial makes such a file from a MicroHH run with the MicroHH processing
chain. We look at what the case provides, export the methane column, check
the file with SPECTRA's validator and pick snapshots for a run file.

The case is the 290 kg/h methane release of the controlled-release
experiment at Pau on 18 September 2024, simulated on a 1 km x 1 km domain
at 1 m resolution. It is the plume that the embedding notebooks use.

You need:

- the MicroHH case directory with the `.ini` file, the online column
  cross-section `ch4_path.xy.nc`, the input file `<case>_input.nc` (it
  holds the time-dependent source strength), the statistics file
  `<case>.default.<starttime>.nc` and `<case>_sources.nc` with the
  geographic position of the release;
- `microhh_processing_chain`, installed separately: SPECTRA does not
  depend on it.

The paths below are placeholders: replace them with your files.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import microhh_processing_chain.initialisation as init
import microhh_processing_chain.io as mio
import numpy as np
import pandas as pd
import xarray as xr
import yaml
from microhh_processing_chain.export import export_plume_column

from spectra.contracts import (
    convert_column_units,
    plume_support_bounds,
    select_time,
    validate_plume_column,
)
from spectra.plume.io import open_plume_column

CASE_DIR = Path("/capstor/store/cscs/userlab/lp171/smeier/microhh_output")
CASE_NAME = "pau_controlled_release"
SUFFIX = "20240918_133000_290kgh_15min_v2"
# UTC time of model second zero: the Pau cases start 1 h 15 min before the
# release, which began at 13:30 UTC.
START_TIME = "2024-09-18T12:15:00"
OUTPUT = Path("/ritom/scratch/cscs/smeier/plume_nc/plumecolumn.nc")

## 1. The MicroHH case

`read_case` takes the case directory, named `<case name>.<suffix>`, and
reads the `.ini` file in it. The export reads these files of the case:

In [ ]:
case = init.read_case(
    case_directory=CASE_DIR / f"{CASE_NAME}.{SUFFIX}_release",
    scratch_path=CASE_DIR,
)
print(case.full_name)
for path in (
    case.ini_file_path,
    case.directory / "ch4_path.xy.nc",
    case.initial_conditions_file_path,
    case.statistics_file_path,
):
    print(f"  {path.name}: {path.stat().st_size / 1e6:.1f} MB")

The `.ini` settings the export relies on: the grid gives the cell size
(here 1 m), `[cross]` writes the column `ch4_path` every 20 s, and
`[source]` places the source in model metres. The source emits a mass
mixing ratio (`swvmr=0`), so `ch4_path` is a mass column. `strength=0`
together with `swtimedep_strength=1` means that the emission rate is a
time series in the input file. `[time]` has no `datetime_utc`, so the UTC
time of model second zero has to be given (`START_TIME`).

In [ ]:
config = case.config
grid = config["grid"]
print("grid:", {key: grid[key] for key in ("itot", "jtot", "ktot", "xsize", "ysize")})
print(f"cell size: {grid['xsize'] / grid['itot']} m x {grid['ysize'] / grid['jtot']} m")
print("time:", {key: config["time"][key] for key in ("starttime", "endtime")})
print("cross:", config["cross"])
keys = (
    "sourcelist",
    "source_x0",
    "source_y0",
    "source_z0",
    "sigma_x",
    "sigma_y",
    "sigma_z",
    "strength",
    "swvmr",
    "swtimedep_strength",
)
print("source:", {key: config["source"][key] for key in keys})
print("datetime_utc:", config.get("time", "datetime_utc"))

## 2. The online column

MicroHH integrates the tracer while it runs: `ch4_path` is
sum(rho_ref q dz) over the full domain depth in kg m-2, on the cell
centres `(y, x)`, with the time in model seconds. The export copies it
unchanged.

In [ ]:
raw = mio.open_microhh_nc(case, "ch4_path.xy")
raw

With `START_TIME` as model second zero, the output covers the release,
which started at 13:30 UTC (model second 4500), and its end at 14:17 UTC:

In [ ]:
seconds = raw["time"].values.astype(float)
start = pd.Timestamp(START_TIME)
print(f"{seconds.size} snapshots every {seconds[1] - seconds[0]:g} s")
print("first:", start + pd.Timedelta(seconds=seconds[0]))
print("last: ", start + pd.Timedelta(seconds=seconds[-1]))

## 3. Exporting the plume column

`export_plume_column` turns the cross-section into a `PlumeColumn`:

- `column(time, y, x)` in kg m-2 (float32), copied from `ch4_path`;
- the coordinates `simulation_time` (model seconds) and `time` (UTC, from
  `START_TIME`);
- `emission_rate(time)` in kg s-1 from the time-dependent source strength;
- `wind_u` and `wind_v(time)`, the horizontal-mean wind at the source
  height from the statistics file;
- attributes with the source position and size, the cell size, the
  reference time and the provenance (MicroHH version, producer, files).

It writes compressed NetCDF to `OUTPUT` (`overwrite=True` replaces an
earlier export) and returns the dataset.

In [ ]:
plume = export_plume_column(
    case, "ch4", start_time=START_TIME, output_path=OUTPUT, overwrite=True
)
plume

## 4. Validating with SPECTRA

SPECTRA validates a plume file when it opens it, as the embedding pipeline
does: dimensions, units, a uniform grid that agrees with `dx` and `dy`, a
UTC time axis, finite non-negative columns and the required attributes.
`validate_plume_column` returns a summary. On the command line,
`spectra validate <file> --contract plume-column` runs the same check.

In [ ]:
plume = open_plume_column(OUTPUT)
info = validate_plume_column(plume)
info

The attributes record where the file came from:

In [ ]:
for key in (
    "case_id",
    "source_id",
    "les_model",
    "les_model_version",
    "producer",
    "producer_version",
    "reference_time",
    "column_source",
    "source_files",
):
    print(f"{key}: {plume.attrs[key]}")

## 5. A look at the plume

The emission rate is zero before the release and after its end. The wind
is the horizontal mean at the source height. The mass in the domain,
sum(column) dx dy, grows for a few minutes after the release starts, until
the plume reaches the open boundaries. It then follows the wind speed,
since slower wind keeps the methane longer in the domain, and falls to
zero after the release ends.

In [ ]:
cell_area = float(plume.attrs["dx"]) * float(plume.attrs["dy"])
mass = plume["column"].sum(("y", "x"), dtype="float64") * cell_area
speed = np.hypot(plume["wind_u"], plume["wind_v"])
# Meteorological convention: the direction the wind blows from.
direction = np.degrees(np.arctan2(-plume["wind_u"], -plume["wind_v"])) % 360

fig, axes = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
axes[0].plot(plume["time"], plume["emission_rate"] * 3600)
axes[0].set_ylabel(r"emission [kg h$^{-1}$]")
axes[1].plot(plume["time"], speed)
axes[1].set_ylabel(r"wind speed [m s$^{-1}$]")
twin = axes[1].twinx()
twin.plot(plume["time"], direction, "C1.", ms=3)
twin.set_ylabel(r"wind direction [$^\circ$]", color="C1")
axes[2].plot(plume["time"], mass)
axes[2].set_ylabel("mass in domain [kg]")
axes[2].set_xlabel("time [UTC]")
fig.autofmt_xdate()

Three snapshots of the column in ppm-m during the release. Snapshot 120
(13:55 UTC) is the one the embedding notebooks use; the cyan star marks
the emission source.

In [ ]:
TIME_INDICES = (60, 120, 180)
fig, axes = plt.subplots(1, len(TIME_INDICES), figsize=(15, 4.2), sharey=True)
for ax, index in zip(axes, TIME_INDICES, strict=True):
    snapshot = select_time(plume, index)
    amount = convert_column_units(snapshot["column"], "ppm-m", gas="ch4")
    amount.plot.imshow(
        ax=ax,
        cmap="magma",
        robust=True,
        cbar_kwargs={"label": r"CH$_4$ vertical column [ppm-m]"},
    )
    ax.plot(plume.attrs["source_x"], plume.attrs["source_y"], "c*", ms=12)
    ax.set_title(f"{index}: {snapshot.attrs['snapshot_time']} UTC")

The `support_bounds_m` of the summary above cover the whole domain: away
from the plume the column is tiny but not exactly zero. The footprint of
a snapshot is therefore taken above a threshold, here 1 ppm-m. Its extent
in model metres shows how much room the plume needs in a scene:

In [ ]:
one_ppm_m = xr.DataArray(1.0, attrs={"units": "ppm-m"})
threshold = float(convert_column_units(one_ppm_m, "kg m-2", gas="ch4"))
print(f"1 ppm-m = {threshold:.3g} kg m-2")
for index in TIME_INDICES:
    xmin, ymin, xmax, ymax = plume_support_bounds(
        select_time(plume, index), threshold=threshold
    )
    print(f"{index}: x {xmin:.0f} to {xmax:.0f} m, y {ymin:.0f} to {ymax:.0f} m")

## 6. Using the plume in a run file

A run file embeds one snapshot per plume entry. Its anchor pins the
emission source (`source_x`, `source_y`) by default, so anchoring at the
release site puts the plume where the experiment released it. Snapshots
from about five minutes after the release start until its end (13:35 to
14:17 UTC) show a plume that has crossed the domain, at the full emission
rate:

In [ ]:
candidates = list(range(60, 181, 20))
table = pd.DataFrame(
    {
        "time": plume["time"].values[candidates],
        "simulation_time_s": plume["simulation_time"].values[candidates],
        "emission_kg_h": plume["emission_rate"].values[candidates] * 3600,
        "mass_in_domain_kg": mass.values[candidates],
    },
    index=pd.Index(candidates, name="time_index"),
)
table.round({"emission_kg_h": 1, "mass_in_domain_kg": 1})

The sources file of the case holds the geographic position of the release
at the model position `source_x0`, `source_y0`:

In [ ]:
with xr.open_dataset(case.directory / f"{case.name}_sources.nc") as sources:
    release = sources.isel(source=0).load()
print(
    "model position:",
    (float(release["source_x0"]), float(release["source_y0"])),
    "plume file:",
    (float(plume.attrs["source_x"]), float(plume.attrs["source_y"])),
)
entry = {
    "file": str(OUTPUT),
    "time_index": 120,
    "anchor": {"lon": float(release["lon"]), "lat": float(release["lat"])},
}
print(yaml.safe_dump({"plumes": [entry]}, sort_keys=False))

## Notes

- `tracer=` selects another MicroHH scalar and `source_index=` the source
  that emits it. A tracer emitted by several sources is refused, since a
  plume file holds a single source.
- `times=` exports a subset of snapshots, given in model seconds, which
  keeps files small.
- `source="3d"` integrates the 3-D tracer field instead of reading the
  cross-section, and `altitudes_m=` then adds partial columns
  (`column_below_altitude`). It needs the potential temperature `th`,
  which this case does not store.
- The file stays in the model frame and carries no map projection: the run
  file places, rotates and flips the plume in the scene.
- Cases whose `.ini` sets `[time] datetime_utc` need no `start_time`.
- The checks above read the whole column (about 0.9 GB for this case); the
  notebook needs about 4 GB of memory.